# MBG Full-Book Extraction Test (Qwen)
This notebook runs MBG extraction for the full book using the precomputed OCR artifacts in `artifacts/deepseek_ocr_fullbook`.

In [ ]:
from pathlib import Path
import json
import subprocess

ROOT = Path('/home/snt/projects/GramMetaRL')
OCR_DIR = ROOT / 'artifacts/deepseek_ocr_fullbook'
OCR_PAGES_JSONL = OCR_DIR / 'pages_ocr.jsonl'
EXTRACT_MODE = 'block'  # chapter | page_batch | block
BATCH_PAGES = 3
TEST_DIR = ROOT / f'artifacts/mbg_fullbook_qwen_test_{EXTRACT_MODE}'
TEST_DIR.mkdir(parents=True, exist_ok=True)

MBG_OUT = TEST_DIR / 'lb_mbg_cards_fullbook.jsonl'
LLM_ENDPOINT = 'http://10.6.32.16:8000/v1'
LLM_MODEL = 'nvidia/Qwen3.6-35B-A3B-NVFP4'

page_numbers = []
with OCR_PAGES_JSONL.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if row.get('error'):
            continue
        page = row.get('page_number')
        if isinstance(page, int):
            page_numbers.append(page)

page_numbers = sorted(set(page_numbers))
if not page_numbers:
    raise ValueError('No valid pages found in OCR pages JSONL')

PAGE_START = page_numbers[0]
PAGE_END = page_numbers[-1]

print('OCR dir:', OCR_DIR)
print('OCR pages JSONL:', OCR_PAGES_JSONL)
print('Page count found:', len(page_numbers))
print('Page range:', page_numbers[0], '-', page_numbers[-1])
print('Extract mode:', EXTRACT_MODE)
print('Batch pages:', BATCH_PAGES)
print('Test dir:', TEST_DIR)
print('Output JSONL:', MBG_OUT)

OCR dir: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook
OCR pages JSONL: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook/pages_ocr.jsonl
Page count found: 74
Page range: 1 - 74
Extract mode: block
Batch pages: 3
Test dir: /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test_block
Output JSONL: /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test_block/lb_mbg_cards_fullbook.jsonl


In [6]:
# Run MBG extraction with the selected granularity mode (single JSONL output)
cmd = [
    str(ROOT / '.venv/bin/python'),
    str(ROOT / 'main.py'),
    'build-mbg',
    '--output', str(MBG_OUT),
    '--language', 'lb',
    '--source-id', 'luxembourgish_grammar_book_fullbook',
    '--ocr-pages-dir', str(OCR_DIR),
    '--page-start', str(PAGE_START),
    '--page-end', str(PAGE_END),
    '--extract-mode', EXTRACT_MODE,
    '--llm-endpoint', LLM_ENDPOINT,
    '--llm-model', LLM_MODEL,
    '--llm-timeout', '900',
    '--max-chunk-chars', '2000',
]
if EXTRACT_MODE == 'page_batch':
    cmd.extend(['--batch-pages', str(BATCH_PAGES)])

print('Running command:')
print(' '.join(cmd))
proc = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

captured_lines = []
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
    captured_lines.append(line)

ret = proc.wait()
print('\nReturn code:', ret)
if ret != 0:
    tail = ''.join(captured_lines[-120:])
    print('\n--- LAST LOG LINES ---')
    print(tail)
    raise RuntimeError('build-mbg failed')

Running command:
/home/snt/projects/GramMetaRL/.venv/bin/python /home/snt/projects/GramMetaRL/main.py build-mbg --output /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test_block/lb_mbg_cards_fullbook.jsonl --language lb --source-id luxembourgish_grammar_book_fullbook --ocr-pages-dir /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook --page-start 1 --page-end 74 --extract-mode block --llm-endpoint http://10.6.32.16:8000/v1 --llm-model nvidia/Qwen3.6-35B-A3B-NVFP4 --llm-timeout 900 --max-chunk-chars 2000
[MBG] section 1/394 title=## Summary pages=1-1
[MBG] section 1/394 extracted=0 added=0
[MBG] section 2/394 title=## Keywords pages=2-2
[MBG] section 2/394 extracted=0 added=0
[MBG] section 3/394 title=## 1. Introduction pages=2-2
[MBG] section 3/394 extracted=0 added=0
[MBG] section 4/394 title=## 1. Introduction pages=3-3
[MBG] section 4/394 extracted=0 added=0
[MBG] section 5/394 title=## [COMP: FIGURE 1 HERE] pages=3-3
[MBG] section 5/394 extracted=0 added=0
[MB

In [7]:
# Inspect full-book extraction outputs
cards = []
with MBG_OUT.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            cards.append(json.loads(line))

print('Output JSONL:', MBG_OUT)
print('Total cards:', len(cards))

for i, c in enumerate(cards[:10], start=1):
    print(f'\n[{i}] id={c.get("id")}, scope={c.get("scope")}, op={c.get("operation_type")}, priority={c.get("priority")}')
    print('tags:', c.get('phenomenon_tags', []))
    print('triggers:', c.get('trigger_conditions', [])[:2])
    src = c.get('source', {})
    print('source pages:', src.get('page_start'), '-', src.get('page_end'), '| section:', src.get('section_title'))

Output JSONL: /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test_block/lb_mbg_cards_fullbook.jsonl
Total cards: 362

[1] id=AUTO_9e5ef21d38c7, scope=unspecified, op=fallback, priority=40
tags: ['grammar_rule']
triggers: ['When analyzing closed or close-mid monophthongs (e.g., /i/, /e/, /y/).']
source pages: 10 - 10 | section: ## 3. Phonetics and Phonology

[2] id=AUTO_8e0750839bc8, scope=unspecified, op=fallback, priority=70
tags: ['word_order']
triggers: ['When encountering long [e:] in loanwords.']
source pages: 10 - 10 | section: ## 3. Phonetics and Phonology

[3] id=AUTO_35bf521da603, scope=unspecified, op=fallback, priority=60
tags: ['grammar_rule']
triggers: ['When schwa occurs in a stressed syllable.']
source pages: 10 - 10 | section: ## 3. Phonetics and Phonology

[4] id=AUTO_a520aae586c8, scope=unspecified, op=fallback, priority=50
tags: ['grammar_rule']
triggers: ['When analyzing the short near-open vowel [æ].']
source pages: 10 - 10 | section: ## 3. Phonetics and

In [8]:
# Example Agent smoke test on a known grammar-dense slice
EXAMPLE_MBG_OUT = TEST_DIR / 'lb_mbg_cards_p36_p40_with_examples.jsonl'
EXAMPLE_OUT = TEST_DIR / 'lb_mbg_cards_p36_p40_examples.jsonl'

cmd = [
    str(ROOT / '.venv/bin/python'),
    str(ROOT / 'main.py'),
    'build-mbg',
    '--output', str(EXAMPLE_MBG_OUT),
    '--language', 'lb',
    '--source-id', 'luxembourgish_grammar_book_p36_p40_examples',
    '--ocr-pages-dir', str(OCR_DIR),
    '--page-start', '36',
    '--page-end', '40',
    '--enable-example-agent',
    '--example-agent-output', str(EXAMPLE_OUT),
    '--llm-endpoint', LLM_ENDPOINT,
    '--llm-model', LLM_MODEL,
    '--llm-timeout', '900',
    '--max-chunk-chars', '2000',
]

print('Running command:')
print(' '.join(cmd))
proc = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

captured_lines = []
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')
    captured_lines.append(line)

ret = proc.wait()
print('\nReturn code:', ret)
if ret != 0:
    tail = ''.join(captured_lines[-120:])
    print('\n--- LAST LOG LINES ---')
    print(tail)
    raise RuntimeError('build-mbg with example agent failed')

print('MBG output:', EXAMPLE_MBG_OUT)
print('Example sidecar:', EXAMPLE_OUT)

Running command:
/home/snt/projects/GramMetaRL/.venv/bin/python /home/snt/projects/GramMetaRL/main.py build-mbg --output /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test_block/lb_mbg_cards_p36_p40_with_examples.jsonl --language lb --source-id luxembourgish_grammar_book_p36_p40_examples --ocr-pages-dir /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook --page-start 36 --page-end 40 --enable-example-agent --example-agent-output /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test_block/lb_mbg_cards_p36_p40_examples.jsonl --llm-endpoint http://10.6.32.16:8000/v1 --llm-model nvidia/Qwen3.6-35B-A3B-NVFP4 --llm-timeout 900 --max-chunk-chars 2000
[MBG] section 1/2 title=## 4.6 Prepositions pages=36-37
[MBG] section 1/2 extracted=5 added=5
[MBG] section 2/2 title=## 4.7 Verbs pages=37-40
[MBG] section 2/2 extracted=10 added=10
[MBG] completed sections=2 cards=15 output=/home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test_block/lb_mbg_cards_p36_p40_w